In [0]:
"""
01_machine_kpis.py

Machine Performance KPIs

Source:
    fact_operations

Target:
    machine_kpis

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    count,
    current_timestamp,
    sum,
    when,
)

from pyspark.sql.functions import col

# ============================================================
# Machine KPIs
# ============================================================

@dlt.table(
    name="machine_kpis",
    comment="Machine performance KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def machine_kpis():

    operations = dlt.read("fact_operations")

    return (

        operations

        .groupBy(

            "plant_code",
            "hall_id",
            "line_id",
            "machine_id",
            "machine_name",
            "machine_type",

        )

        .agg(

            count("*").alias(
                "operations_completed"
            ),

            avg(
                "cycle_time_sec"
            ).alias(
                "average_cycle_time_sec"
            ),

            avg(
                "actual_force_kn"
            ).alias(
                "average_force_kn"
            ),

            avg(
                "force_deviation_kn"
            ).alias(
                "average_force_deviation_kn"
            ),

            sum(

                when(

                    operations.quality_result == "PASS",

                    1

                ).otherwise(0)

            ).alias(
                "passed_operations"
            ),

            sum(

                when(

                    operations.quality_result != "PASS",

                    1

                ).otherwise(0)

            ).alias(
                "failed_operations"
            ),

        )

        .withColumn(

            "pass_rate",

            (
                col("passed_operations")
                * 100.0
                / col("operations_completed")
            )

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )